
## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

## DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")  
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


In [5]:
## pydantic
from pydantic import BaseModel, Field

class Player(BaseModel):
    name:str=Field(description="The name of the player")
    age:int=Field(description="The age of the player")
    goals:int=Field(description="The number of goals scored by the player")
    assists:int=Field(description="The number of assists made by the player")
    nationality:str=Field(description="The nationality of the player")
    position:str=Field(description="The position of the player on the field")

class Club(BaseModel):
    name:str=Field(description="The name of the club")
    country:str=Field(description="The country where the club is based")
    league:str=Field(description="The league in which the club competes")
    players:list[Player]=Field(description="A list of players in the club")
    trophies:int=Field(description="The number of trophies won by the club")

model_with_structure=model.with_structured_output(Club)

response=model_with_structure.invoke("Provide information about the football club FC Barcelona, including its players and their statistics.")

response


Club(name='FC Barcelona', country='Spain', league='La Liga', players=[Player(name='Robert Lewandowski', age=35, goals=20, assists=8, nationality='Poland', position='Forward'), Player(name='Pedri', age=21, goals=4, assists=6, nationality='Spain', position='Midfielder'), Player(name='Ronald Araujo', age=25, goals=1, assists=0, nationality='Uruguay', position='Defender'), Player(name='Marc-Andre ter Stegen', age=32, goals=0, assists=0, nationality='Germany', position='Goalkeeper')], trophies=98)

In [7]:

class Actor(BaseModel):
    name:str
    age:int
    nationality:str

class Movie(BaseModel):
    name:str
    budget:int
    actors:list[Actor]
    cast:list[Actor]

model_with_st=model.with_structured_output(Movie)

response=model_with_st.invoke("Provide information about the movie Inception, including its actors and their roles.")
response

Movie(name='Inception', budget=160000000, actors=[Actor(name='Leonardo DiCaprio', age=49, nationality='American'), Actor(name='Joseph Gordon-Levitt', age=43, nationality='American'), Actor(name='Elliot Page', age=37, nationality='Canadian')], cast=[Actor(name='Leonardo DiCaprio', age=49, nationality='American'), Actor(name='Joseph Gordon-Levitt', age=43, nationality='American'), Actor(name='Elliot Page', age=37, nationality='Canadian')])

In [8]:
## TypedDict
from typing import TypedDict,Annotated
class ActorDict(TypedDict):
    name:Annotated[str,"The name of the actor"]
    age:Annotated[int,"The age of the actor"]
    nationality:Annotated[str,"The nationality of the actor"]

class MovieDict(TypedDict):
    name:Annotated[str, "The name of the movie"]
    budget:Annotated[int, "The budget of the movie in USD"]
    actors:Annotated[list[ActorDict], "A list of actors in the movie"]
    cast:Annotated[list[ActorDict], "A list of cast members in the movie"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'name': 'The Avengers',
 'budget': 220000000,
 'actors': [{'name': 'Robert Downey Jr.',
   'age': 57,
   'nationality': 'American'},
  {'name': 'Chris Evans', 'age': 41, 'nationality': 'American'},
  {'name': 'Scarlett Johansson', 'age': 38, 'nationality': 'American'}]}

In [9]:
model.profile

{'name': 'Gemini 3.5 Flash Lite',
 'release_date': '2026-07-21',
 'last_updated': '2026-07-21',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

In [15]:
## DataClasses
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class person:
    name:str
    age:int
    email:str
    phone:str

agent=create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=person
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Please provide the details of the person named Susovan Paul, including his age(22), palsusovan88@gmail.com, and phone number(8910760697)."}]
})
result


{'messages': [HumanMessage(content='Please provide the details of the person named Susovan Paul, including his age(22), palsusovan88@gmail.com, and phone number(8910760697).', additional_kwargs={}, response_metadata={}, id='8c592843-8a6e-47f6-b1ef-002c589a1c03'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "Susovan Paul",\n  "age": 22,\n  "email": "palsusovan88@gmail.com",\n  "phone": "8910760697"\n}', 'extras': {'signature': 'EmAKXgFpFH0TQZpPpvghmCURPx+w/xp5Gn8wryaIonnAGjmkIxOcLGz+GH6ZZTKiXMSeaKNC8CyNQ3eRwp7eESLS3s8xvnCzTbWINYHWVqXB78tV7kuIQEQ/JbtEid1lcJ8='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a11b72-f621-7f21-8c88-2ab8eac85d8a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 56, 'total_tokens': 101, 'input_token_details': {'cache_read': 0}})],
 'structured_response': person(name='Su

In [16]:
result["structured_response"]

person(name='Susovan Paul', age=22, email='palsusovan88@gmail.com', phone='8910760697')